# Spark ETL — flights from Kafka to results

The **core processing stage**: `Kafka (group10.flights) -> Spark -> clean -> filter commercial -> join reference data -> aggregate -> parquet`.

It computes **all four analysis questions** and stores each result as parquet:
- **Q1 Dominance** — which airlines / carrier nations fly the most over Europe?
- **Q2 Geography** — where are the densest air corridors? (flight-density grid)
- **Q3 Busiest airports** — low climbing/descending aircraft matched to their nearest airport (spatial join).
- **Q4a Departures vs arrivals** — split the airport matches by climb/descent.

**Run on the JupyterHub.** It reads **live** from Kafka, so make sure the producer has run recently / is running. The broker has short retention, which is exactly why we persist results to parquet.

## 1. SparkSession
Connect to the cluster master and pull in the Kafka connector (`spark-sql-kafka`, matched to Spark 3.5.6). First run on a fresh kernel downloads the jars via Ivy (one-time).

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("group10-flights-etl")
    .master("spark://172.29.16.102:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| app", spark.sparkContext.applicationId)

## 2. Read from Kafka and parse the JSON
Batch-read the whole topic (`startingOffsets=earliest`) and parse the JSON into typed columns. We `.cache()` so the (live, growing) topic is read **once** and every downstream step sees the same snapshot.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType, LongType,
)

schema = StructType([
    StructField("icao24", StringType()),
    StructField("callsign", StringType()),
    StructField("origin_country", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("baro_altitude", DoubleType()),
    StructField("on_ground", BooleanType()),
    StructField("velocity", DoubleType()),
    StructField("true_track", DoubleType()),
    StructField("vertical_rate", DoubleType()),
    StructField("geo_altitude", DoubleType()),
    StructField("snapshot_time", LongType()),
])

raw = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "172.29.16.101:9092")
    .option("subscribe", "group10.flights")
    .option("startingOffsets", "earliest")
    .load())

# cache() pins ONE consistent read; without it each later action re-reads the
# growing topic (producer is live) -> inconsistent counts.
flights = raw.select(F.from_json(F.col("value").cast("string"), schema).alias("f")).select("f.*").cache()
print("parsed flights:", flights.count())  # first action fills the cache

## 3. Clean
Keep airborne aircraft with a callsign; derive `airline_icao` (first 3 chars of the callsign).

In [ ]:
clean = (flights
    .filter(~F.col("on_ground"))
    .filter(F.col("callsign") != "")
    .withColumn("airline_icao", F.substring("callsign", 1, 3)))
print("airborne w/ callsign:", clean.count())

## 4. Keep only commercial flights (broadcast join with the airline reference)
Load the scraped airline-codes lookup (Wikipedia, source 2) from the repo on GitHub, broadcast it, and **inner-join** on the ICAO code. The inner join is the **commercial filter** — private/military/GA callsigns drop out. We cache the result since every question below uses it.

In [ ]:
import pandas as pd

AIRLINES_CSV = "https://raw.githubusercontent.com/jakobiene/bde-project-group10/main/data/processed/airlines.csv"
airlines = spark.createDataFrame(pd.read_csv(AIRLINES_CSV))

commercial = clean.join(F.broadcast(airlines), clean.airline_icao == airlines.icao, "inner").cache()
print("commercial flights:", commercial.count())

## 5. Q1 — Dominance
Distinct aircraft per airline and per operator country, then stored to parquet. (Results are small, so we collect to the driver and write locally — avoids distributed-write issues on the shared cluster.)

In [ ]:
by_airline = (commercial.groupBy("airline_icao", "airline", "country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))

by_country = (commercial.groupBy("origin_country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))

by_airline.show(10, truncate=False)
by_country.show(10, truncate=False)

In [ ]:
import os
# notebook runs from notebooks/, so go up one level into the repo-root data/processed
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)
by_airline.toPandas().to_parquet(f"{OUT_DIR}/dominance_by_airline.parquet", index=False)
by_country.toPandas().to_parquet(f"{OUT_DIR}/dominance_by_country.parquet", index=False)
print("Q1 results written to", os.path.abspath(OUT_DIR))

## 6. Q2 — Geography: flight-density grid (corridors)
Round each position to a ~0.1 deg grid (~11 km cells) and count the flight records per cell. The busiest cells reveal the densest **air corridors** over Europe — this parquet feeds a heatmap in the visualisation notebook.

(Counting records, not distinct aircraft, is intentional: a cell that many flights pass through over time should score high.)

In [ ]:
density = (commercial
    .withColumn("lat_bin", F.round("latitude", 1))
    .withColumn("lon_bin", F.round("longitude", 1))
    .groupBy("lat_bin", "lon_bin")
    .agg(F.count("*").alias("flights"))
    .orderBy(F.desc("flights")))

density.show(10)
density.toPandas().to_parquet(f"{OUT_DIR}/density_grid.parquet", index=False)
print("Q2 density grid written")

## 7. Q3 — Busiest airports (spatial join)
Load the airports reference (OurAirports, source 3), then pick **arrival/departure candidates**: aircraft that are **low (<3000 m) and climbing/descending** (`|vertical_rate| > 1`). Such aircraft must be near an airport (also drops negative-altitude outliers).

In [ ]:
AIRPORTS_CSV = "https://raw.githubusercontent.com/jakobiene/bde-project-group10/main/data/processed/airports.csv"
airports = spark.createDataFrame(pd.read_csv(AIRPORTS_CSV))
print("airports:", airports.count())

arrdep = (commercial
    .filter(F.col("geo_altitude").isNotNull())
    .filter((F.col("geo_altitude") >= 0) & (F.col("geo_altitude") < 3000))
    .filter(F.abs(F.col("vertical_rate")) > 1)
    .select("icao24", "callsign", "latitude", "longitude", "vertical_rate", "geo_altitude"))
print("arrival/departure candidates:", arrdep.count())

Cross-join each candidate with all airports, compute the **haversine distance**, keep the **nearest** airport within **30 km**. We cache `nearest` because Q3 and Q4a both use it. Counting distinct aircraft per airport gives the busiest-airports ranking.

In [ ]:
from pyspark.sql.window import Window
R = 6371.0  # earth radius, km

ap = airports.select(
    F.col("icao_code"), F.col("name").alias("airport_name"),
    F.col("iso_country"), F.col("latitude_deg"), F.col("longitude_deg"))

pairs = arrdep.crossJoin(F.broadcast(ap)).withColumn(
    "dist_km",
    2 * R * F.asin(F.sqrt(
        F.pow(F.sin(F.radians(F.col("latitude_deg") - F.col("latitude")) / 2), 2)
        + F.cos(F.radians(F.col("latitude"))) * F.cos(F.radians(F.col("latitude_deg")))
        * F.pow(F.sin(F.radians(F.col("longitude_deg") - F.col("longitude")) / 2), 2)
    )))

w = Window.partitionBy("icao24").orderBy("dist_km")
nearest = (pairs.withColumn("rk", F.row_number().over(w))
    .filter(F.col("rk") == 1)
    .filter(F.col("dist_km") <= 30)).cache()

busiest = (nearest.groupBy("icao_code", "airport_name", "iso_country")
    .agg(F.countDistinct("icao24").alias("aircraft"))
    .orderBy(F.desc("aircraft")))
busiest.show(15, truncate=False)
busiest.toPandas().to_parquet(f"{OUT_DIR}/busiest_airports.parquet", index=False)
print("Q3 results written")

## 8. Q4a — Departures vs arrivals per airport
Reuse the nearest-airport matches and split by flight phase: **climbing (`vertical_rate > 1`) = just departed**, **descending = arriving**. Gives the top departure and arrival airports separately. (True point-to-point routes A->B would need OpenSky's `/flights` endpoint — left as a stretch goal.)

In [ ]:
labeled = nearest.withColumn(
    "phase", F.when(F.col("vertical_rate") > 1, "departure").otherwise("arrival"))

top_departures = (labeled.filter(F.col("phase") == "departure")
    .groupBy("icao_code", "airport_name", "iso_country")
    .agg(F.countDistinct("icao24").alias("departures"))
    .orderBy(F.desc("departures")))

top_arrivals = (labeled.filter(F.col("phase") == "arrival")
    .groupBy("icao_code", "airport_name", "iso_country")
    .agg(F.countDistinct("icao24").alias("arrivals"))
    .orderBy(F.desc("arrivals")))

print("Top departure airports:")
top_departures.show(10, truncate=False)
print("Top arrival airports:")
top_arrivals.show(10, truncate=False)

top_departures.toPandas().to_parquet(f"{OUT_DIR}/top_departures.parquet", index=False)
top_arrivals.toPandas().to_parquet(f"{OUT_DIR}/top_arrivals.parquet", index=False)
print("Q4a results written")

## Summary
This notebook is the **Kafka -> Spark -> storage** ETL. From the live `group10.flights` stream it produced and stored to `data/processed/`:
- `dominance_by_airline`, `dominance_by_country` (Q1)
- `density_grid` (Q2 corridors)
- `busiest_airports` (Q3)
- `top_departures`, `top_arrivals` (Q4a)

The visualisation notebook (05) reads these parquet files and builds the charts / map. The remaining pieces are the **"whose airspace" country attribution** (Q2, done with `reverse_geocoder` in pandas) and, optionally, **true routes A->B** via OpenSky's `/flights` endpoint (Q4 stretch).